# Laboratorio 10

- Mathew Alexander Cordero Aquino 22982
- Pedro Pablo Guzman Mayen  22111



[Repositorio Lab 10](https://github.com/donmatthiuz/DeepLearning/tree/lab10)

## SHAP (SHapley Additive exPlanations)

### Concepto

Shapely es un metodo que viene de la teoria de juegos que se encarga de analizar modelos.

Cada feature del modelo es visto como un jugador que aporta a la prediccion. 

Esto hace que se basa en 3 ideas

#### Teoria de juegos: 
Se distribuyen las ganancias (prediccion del modelo) entre jugadores  (features)

#### Modelos aditivos: 

La siguiente ecuacion explica esta parte.

$$
\hat{y} = \phi_0 + \sum_{i=1}^{n} \phi_i
$$


* $\phi_0$: valor esperado del modelo (mean prediction)
* $\phi_i$: contribución de cada característica



#### Contribución marginal

SHAP analiza todas las combinaciones posibles de features (subconjuntos) y mide cuánto cambia la predicción cuando una feature entra o sale del “equipo”.




### Tipos de modelos a los que aplica

| Tipo de modelo                                           | Método SHAP usado   | Descripción                                   |
| -------------------------------------------------------- | ------------------- | --------------------------------------------- |
| **Árboles (XGBoost, LightGBM, CatBoost, Random Forest)** | **TreeSHAP**        | Rápido y exacto.                              |
| **Modelos lineales**                                     | **LinearExplainer** | Usa pesos lineales directamente.              |
| **Redes neuronales**                                     | **DeepSHAP**        | Basado en DeepLIFT + Shapley.                 |
| **Modelos arbitrarios (black-box)**                      | **KernelSHAP**      | Basado en muestreo; más lento pero universal. |


### Ejemplo

Tenemos un modelo de arbol tipo XGBoost , el modelo tiene los features de :

- tamaño (m²)
- número de cuartos
- años de antigüedad
- distancia al centro
Que hace predicciones del precio de una casa. Y para la misma se hace la prediccion de

| Feature          | Valor |
| ---------------- | ----- |
| m²               | 120   |
| Cuartos          | 3     |
| Antigüedad       | 20    |
| Distancia centro | 15 km |

El codigo seria para describir las features son los siguientes

```python
import xgboost as xgb
import shap

# Entrenar modelo
model = xgb.XGBRegressor()
model.fit(X_train, y_train)

# Crear el explainer
explainer = shap.TreeExplainer(model)

# Tomar una muestra
instance = X_test.iloc[0:1]

# Calcular valores SHAP
shap_values = explainer(instance)

# Mostrar gráfico de explicación local
shap.plots.waterfall(shap_values[0])
```


Supongamos que la predicción del modelo es Q120,000.

Entonces los features diran lo siguiente

* `+35,000`: por tener **120 m²**
* `+15,000`: por tener **3 cuartos**
* `–10,000`: porque tiene **20 años**
* `–20,000`: porque está **lejos del centro (15 km)**

Y el valor base del modelo (promedio) es **Q100,000**.

[
100,000 + 35,000 + 15,000 - 10,000 - 20,000 = 120,000
]


Sabemos que cada feature le da o le resta quetzales a la casa. Lo cual hace que valga menos o mas

## Lime


### Concepto

Lo que hace LIME es describir que hace un modelo con su prediccion , construyendo un modelo en base a eso que se pueda entender. Osea de todo el conjunto de predicciones se queda con un local que el toma y luego en base a ese crea otro modelo mas sencillo que le da un sentido .

Los pasos son los siguientes

- Toma una instancia local como predecir el precio de una sola casa, generacion de una sola imagen, etc.

- Genera perturbaciones en la instancia como creando muestras vecinas, cambiando o eliminando features .

- Evalua el modelo original con esas perturbaciones

- Calcula pesos segun la cercania

- Ajusta el modelo interpretable con los resultados que le dio el original

- Reporta pesos y actualiza el estado de nuestro modelo interpretable

Es como construir un modelo de caja blanco a partir de las predicciones del de caja negra


### Modelos a los que aplica

| Tipo                             | ¿Aplica LIME?          |
| -------------------------------- | ---------------------- |
| Modelos lineales                 | SI                      |
| Árboles (Random Forest, XGBoost) | SI                      |
| SVM                              | SI                      |
| Redes neuronales                 | SI                      |
| Modelos de NLP                   | SI                      |
| Modelos de visión (CNN)          | SI (versión LIME-Image) |
| Modelos tabulares                | SI                      |


### Ejemplo

Supongamos que entrenaste un modelo para predecir si una persona recibirá un préstamo.

Entrada:

- Ingresos: 3,500
- Edad: 28
- Historial crediticio: Malo
- Deuda: 45%
- Antigüedad laboral: 1 año

El modelo predice: rechazado (0.91 prob.)

Cómo LIME lo explica:

- Genera copias modificadas de esta persona (cambiando ingresos, edad, deuda…).

- Pide al modelo complejo su predicción para cada copia.

- Pesa más las copias parecidas.

- Ajusta un modelo lineal local.

Con la interpretacion de 

LIME Explanation - Clase: Rechazado

+0.38 Historial crediticio = Malo

+0.22 Deuda > 40%

+0.15 Antigüedad laboral < 2 años

-0.12 Ingreso > 3000


## Counterfactual Explanations

### Concepto
Lo que hace es predecir qué cambios específicos producirían un resultado distinto.

Esto se basa en 

- Mantenemos la predicción objetivo opuesta
En donde se busca un ejemplo muy parecido pero con resultado completamente diferente

- Buscamos el “mínimo cambio necesario”

Un contrafactual intenta modificar lo menos posible:

* pocas features
* cambios pequeños
* que los valores modificados sean plausibles

Usamos criterios de optimización

La búsqueda del contrafactual suele involucrar una función de costo:

$$
\text{minimizar} \quad \text{Distancia}(x, x') + \lambda \cdot \text{Loss}(f(x'), y_{target})
$$

Donde:

* ( x ) = la instancia actual
* ( x' ) = instancia modificada (el contrafactual)
* ( f(x') ) = predicción del modelo
* ( y_{target} ) = predicción deseada

### Modelos a los que aplica

Las explicaciones contrafactuales son **model-agnostic**, es decir, funcionan con cualquier modelo:

| Modelo                           | ¿Aplica? |
| -------------------------------- | -------- |
| Regresión logística              | SI        |
| Random Forest                    | SI        |
| Gradient Boosting                | SI        |
| Redes neuronales                 | SI        |
| SVM                              | SI        |
| Modelos complejos (GBM, XGBoost) | SI        |

Solo requieren acceso al modelo **como caja negra** para consultarle la predicción.

### Ejemplo

Caso: Modelo de crédito predice rechazado.

Entrada del cliente:

| Variable           | Valor |
| ------------------ | ----- |
| Ingresos           | 3,500 |
| Antigüedad laboral | 1 año |
| Deuda              | 45%   |
| Historial          | Malo  |

El modelo predice: Rechazado (0.91).

Contrafactual generado:

Si tu nivel de deuda fuera 35% en lugar de 45%,
y tu antigüedad laboral fuera 2 años en lugar de 1,

el modelo te aprobaría el crédito.



